<a href="https://colab.research.google.com/github/yogeshwenwu/GAN/blob/main/GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import mnist

In [2]:
# Load and preprocess the MNIST dataset
(x_train, _), (_, _) = mnist.load_data()
x_train = x_train / 127.5 - 1.0  # Normalize to the range [-1, 1]
x_train = np.expand_dims(x_train, axis=-1)

11490434/11490434 [==============================] - 0s 0us/step


In [3]:
# Generator model
def build_generator(latent_dim):
    model = models.Sequential()
    model.add(layers.Dense(128 * 7 * 7, input_dim=latent_dim))
    model.add(layers.Reshape((7, 7, 128)))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(128, kernel_size=3, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(64, kernel_size=3, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Conv2D(1, kernel_size=3, padding="same", activation="tanh"))
    return model

In [4]:
# Discriminator model
def build_discriminator(img_shape):
    model = models.Sequential()
    model.add(layers.Conv2D(32, kernel_size=3, strides=2, input_shape=img_shape, padding="same"))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding="same"))
    model.add(layers.ZeroPadding2D(padding=((0,1),(0,1))))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(256, kernel_size=3, strides=1, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

In [5]:
# Build and compile the discriminator
img_shape = x_train.shape[1:]
discriminator = build_discriminator(img_shape)
discriminator.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5), metrics=['accuracy'])

In [6]:
# Build the generator
latent_dim = 100
generator = build_generator(latent_dim)

In [7]:
# Build and compile the combined model
discriminator.trainable = False
z = layers.Input(shape=(latent_dim,))
img = generator(z)
validity = discriminator(img)
combined_model = models.Model(z, validity)
combined_model.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5))

In [8]:
# Training parameters
epochs = 30000
batch_size = 64
save_interval = 1000

In [9]:
# Adversarial ground truths
valid = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

In [10]:
# Training the GAN
for epoch in range(epochs):
    # Train Discriminator
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    imgs = x_train[idx]
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    gen_imgs = generator.predict(noise)

    d_loss_real = discriminator.train_on_batch(imgs, valid)
    d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # Train Generator
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    g_loss = combined_model.train_on_batch(noise, valid)

    # Print progress
    if epoch % 100 == 0:
        print(f"{epoch} [D loss: {d_loss[0]} | D accuracy: {100 * d_loss[1]}] [G loss: {g_loss[0]}]")

    # Save generated images at specified intervals
    if epoch % save_interval == 0:
        save_path = f"generated_images/{epoch}.png"
        gen_imgs = 0.5 * gen_imgs + 0.5  # Rescale values to [0, 1]
        plt.imsave(save_path, gen_imgs[0, :, :, 0], cmap='gray')

2/2 [==============================] - 2s 6ms/step


TypeError: ignored

In [ ]:
# Save the generator model
generator.save("gan_generator.h5")

Below Code


In [13]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import mnist

# Set random seed for reproducibility
np.random.seed(1000)

# Load and preprocess the MNIST dataset
(x_train, _), (_, _) = mnist.load_data()
x_train = x_train / 127.5 - 1.0  # Normalize to the range [-1, 1]
x_train = np.expand_dims(x_train, axis=-1)

# Generator model
def build_generator(latent_dim):
    model = models.Sequential()
    model.add(layers.Dense(128 * 7 * 7, input_dim=latent_dim))
    model.add(layers.Reshape((7, 7, 128)))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(128, kernel_size=3, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.UpSampling2D())
    model.add(layers.Conv2D(64, kernel_size=3, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Conv2D(1, kernel_size=3, padding="same", activation="tanh"))
    return model

# Discriminator model
def build_discriminator(img_shape):
    model = models.Sequential()
    model.add(layers.Conv2D(32, kernel_size=3, strides=2, input_shape=img_shape, padding="same"))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding="same"))
    model.add(layers.ZeroPadding2D(padding=((0,1),(0,1))))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(256, kernel_size=3, strides=1, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

# Build and compile the discriminator
img_shape = x_train.shape[1:]
discriminator = build_discriminator(img_shape)
discriminator.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5), metrics=['accuracy'])

# Build the generator
latent_dim = 100
generator = build_generator(latent_dim)

# Build and compile the combined model
discriminator.trainable = False
z = layers.Input(shape=(latent_dim,))
img = generator(z)
validity = discriminator(img)
combined_model = models.Model(z, validity)
combined_model.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5))

# Training parameters
epochs = 3000
batch_size = 64
save_interval = 700

# Adversarial ground truths
valid = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

# Create directory for saving generated images
if not os.path.exists("generated_images"):
    os.makedirs("generated_images")

# Training the GAN
for epoch in range(epochs):
    # Train Discriminator
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    imgs = x_train[idx]
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    gen_imgs = generator.predict(noise)

    d_loss_real = discriminator.train_on_batch(imgs, valid)
    d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
    d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])

    # Train Generator
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    g_loss = combined_model.train_on_batch(noise, valid)

    # Print progress
    if epoch % 100 == 0:
        print(f"{epoch} [D loss: {d_loss} | D accuracy: {100 * d_loss_real[1]}] [G loss: {g_loss}]")

    # Save generated images at specified intervals
    if epoch % save_interval == 0:
        save_path = f"generated_images/{epoch}.png"
        gen_imgs = 0.5 * gen_imgs + 0.5  # Rescale values to [0, 1]
        plt.imsave(save_path, gen_imgs[0, :, :, 0], cmap='gray')

# Save the generator model
generator.save("gan_generator.keras")



2/2 [==============================] - 0s 4ms/step
0 [D loss: 1.0737244188785553 | D accuracy: 60.9375] [G loss: 0.6174420714378357]
2/2 [==============================] - 0s 5ms/step
100 [D loss: 0.015460398979485035 | D accuracy: 100.0] [G loss: 0.7324723601341248]
2/2 [==============================] - 0s 5ms/step
200 [D loss: 0.005982272792607546 | D accuracy: 100.0] [G loss: 0.9346346855163574]
2/2 [==============================] - 0s 5ms/step
300 [D loss: 0.006677946774289012 | D accuracy: 100.0] [G loss: 1.0419220924377441]
2/2 [==============================] - 0s 5ms/step
400 [D loss: 0.0007840635371394455 | D accuracy: 100.0] [G loss: 2.9648079872131348]
2/2 [==============================] - 0s 5ms/step
500 [D loss: 0.0011287982488283888 | D accuracy: 100.0] [G loss: 3.5227460861206055]
2/2 [==============================] - 0s 4ms/step
600 [D loss: 0.00014770840789424255 | D accuracy: 100.0] [G loss: 3.1248810291290283]
2/2 [==============================] - 0s 5ms/step
70